# Pareto analysis with reconstructed DT ice thickness

This notebook:

- converts the stored objectives back to physical quantities;
- reconstructs the physical DT ice thickness from the MOBO variables;
- produces all six pairwise objective trade-off plots;
- allows the Pareto plots to be coloured by DT ice thickness;
- compares DT ice thickness with drive current, reporting Pearson and Spearman correlations.

Run `db_reader.ipynb` first to (re)generate `results/pareto_dominating_points.csv`. All the underlying logic lives in `iceburner.pareto` / `iceburner.scaling`.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from iceburner import pareto

df = pd.read_csv("../results/pareto_dominating_points.csv")
pareto_df = pareto.prepare_pareto_dataframe(df)

print(f"Total Pareto points: {len(pareto_df)}")
print(f"Valid reconstructed ice geometries: {int(pareto_df['ice_geometry_valid'].sum())}")
print(f"Invalid reconstructed ice geometries: {int((~pareto_df['ice_geometry_valid']).sum())}")

pareto_df[[
    "current_MA", "f_R_outer", "f_Pi", "f_Be",
    "R_outer_mm", "Be_thickness_mm", "DT_ice_thickness_mm",
    "reference_DT_ice_thickness_mm", "delta_DT_ice_thickness_mm", "ice_geometry_valid",
]].head()

## Six pairwise Pareto trade-offs

`iceburner.pareto.plot_pareto_tradeoffs` covers all six panels, including **peak IFAR against current**, coloured by whichever variable is passed as `colour_by`.

In [ ]:
cols = [
    "f_laser", "f_R_outer", "f_Pi", "f_Be", "f_rho", "current",
    "DT_ice_thickness_mm", "delta_DT_ice_thickness_mm",
    "Y_TDT", "Y_rhoRDT", "Y_minus_peak_IFAR", "Y_minus_current",
    "T_DT", "rhoR_DT", "peak_IFAR", "current_MA",
]

for col in cols:
    pareto.plot_pareto_tradeoffs(pareto_df, colour_by=col)

## Relationship between DT ice thickness and current

The solid trend line is only a descriptive linear fit to the Pareto points; the true relationship can be non-linear because the optimiser also changes the geometry multipliers.

In [ ]:
ice_current_results = pareto.plot_ice_thickness_vs_current(pareto_df, colour_by="f_Pi")

### Alternative colouring

Colouring by `f_Be` instead helps separate the effect of current from changes in the Be-thickness multiplier.

In [ ]:
pareto.plot_ice_thickness_vs_current(pareto_df, colour_by="f_Be")